In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

# Maskininlärningsmodeller
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# För normalisering och skalning
from sklearn.preprocessing import StandardScaler

# --- CONFIG ---
INPUT_FILE = 'Network_Traffic_Data_for_DDoS_Detection.csv' 
MODEL_OUTPUT = 'student_model.pkl'
TEST_DATA_OUTPUT = 'skarp_test_data.txt'

def run_advanced_training():
    print("🚀 Startar avancerad EDA och Modelljämförelse...")

    # --- 1. DATA INGESTION & TVÄTT ---
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Hittade inte {INPUT_FILE}")
        return

    df = pd.read_csv(INPUT_FILE, sep=None, engine='python')
    df.columns = df.columns.str.strip().str.replace('"', '').str.replace("'", "")
    df = df.fillna(0)
    df = df.drop_duplicates()

    # --- 2. EDA: ANALYS AV OBALANS ---
    features = ['frame.len', 'udp.length', 'ip.proto']
    target = 'bad_packet'
    
    count_normal = len(df[df[target] == 0])
    count_attack = len(df[df[target] == 1])
    
    print(f"📊 Datapunkter: {count_normal} Normala, {count_attack} Attacker.")

    # --- 3. NORMALISERING OCH SKALNING ---
    # Eftersom Logistic Regression är känslig för stora skillnader i tal 
    # (t.ex. frame.len kan vara 1500 medan ip.proto är 17) så skalar vi datat.
    scaler = StandardScaler()
    X = df[features]
    y = df[target]
    X_scaled = scaler.fit_transform(X)

    # Spara skalaren så att backend kan använda samma skala senare!
    joblib.dump(scaler, 'scaler.pkl')

    # --- 4. DATA SPLIT ---
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # --- 5. MODELLJÄMFÖRELSE ---
    models = {
        "Logistic Regression": LogisticRegression(class_weight='balanced'),
        "Random Forest": RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, random_state=42)
    }

    best_f1 = 0
    best_model = None
    best_model_name = ""

    print("\n🔎 Utvärderar modeller...")
    print("-" * 50)
    
    for name, clf in models.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Vi använder F1-score eftersom det är bättre än Accuracy när man har obalanserad data
        score = f1_score(y_test, y_pred)
        acc = accuracy_score(y_test, y_pred)
        
        print(f"{name}:")
        print(f"   F1-Score: {score:.4f} | Accuracy: {acc:.4f}")
        
        if score > best_f1:
            best_f1 = score
            best_model = clf
            best_model_name = name

    print("-" * 50)
    print(f"🏆 Vinnare: {best_model_name} med F1-Score {best_f1:.4f}")

    # --- 6. EXPORTERA TESTDATA (Oskalad för backendens läsbarhet) ---
    # Vi sparar rådata för att backend ska kunna visa riktiga värden i graferna
    test_raw_X, test_raw_X_hold, test_raw_y, test_raw_y_hold = train_test_split(X, y, test_size=0.2, random_state=42)
    test_export = pd.concat([test_raw_X_hold, test_raw_y_hold], axis=1)
    test_export.to_csv(TEST_DATA_OUTPUT, index=False, quoting=3, escapechar=' ')

    # --- 7. SPARA VINNANDE MODELL ---
    joblib.dump(best_model, MODEL_OUTPUT)
    print(f"✅ Vinnande modell ({best_model_name}) sparad!")

if __name__ == "__main__":
    run_advanced_training()